# MLP Retraining Pipeline — Prod Medallion Data

Retrains the **MLP fault prediction model** on fresh telemetry from `prod_medallion.silver_machine.sensorx_telemetry` (800W devices only)

**Steps:**
1. Read telemetry + fault events, forward-fill fault state, compute delta seconds
2. Engineer 112 features (5 sensors × 20 lags + 6 daily deviations + deviceId index)
3. Chronological 80/20 train/test split → save to `teams.sensorx.train_data` / `test_data`
4. Normalize, train MLP `[112, 16, 16, 2]`, evaluate (AUC, recall, F1)
5. Register model to Unity Catalog (`teams.sensorx.mlp_fault_prediction`) via MLflow

In [0]:
# Load 800W device filter from saved table
import os
import mlflow
from mlflow.models import infer_signature
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from pyspark.sql.window import Window

devices_800w = spark.read.table("teams.sensorx.devices_800w")
print(f"Loaded {devices_800w.count()} 800W device IDs")

In [0]:
def data_engineering_retrain_mlp(days, H_days=15, n_lags=20):
    """
    Read and engineer data from prod_medallion for MLP training.
    Returns chronological train/test split with 112-feature vectors.

    Parameters:
    days : int
        Number of days of historical data to use.
    H_days : int
        Prediction horizon in days (default: 15).
    n_lags : int
        Number of lag features per sensor (default: 20).
    """

    print(f"Reading and engineering data...")
    print(f"  Source: {days} days from prod_medallion")
    print(f"  Horizon: {H_days} days | Lags: {n_lags}")

    #  Read telemetry
    df_telemetry = table_service.load_table_from_device_checkpoint(
        table_name="prod_medallion.silver_machine.sensorx_telemetry",
        machine_name="sensorx",
        max_historical_days=days,
        old_format=True
    ).select(
        "properties_deviceID",
        "properties_payloadTimeStamp",
        "payload_xrayController_filamentCurrent",
        "payload_xrayController_temperature",
        "payload_xrayController_tubeCurrent",
        "payload_xrayController_tubeVoltage",
    )

    # Filter to 800W devices
    df_telemetry_800w = df_telemetry.join(
        F.broadcast(devices_800w),
        df_telemetry["properties_deviceID"] == devices_800w["properties_deviceId"],
        "inner"
    ).drop(devices_800w["properties_deviceId"])

    # Fault forward-fill 
    df_fault = (
        table_service.load_table_from_device_checkpoint(
            table_name="prod_medallion.silver_machine.pluto_xraycontroller_fault_property",
            machine_name="sensorx",
            max_historical_days=days,
            old_format=True
        )
        .filter(F.col("payload_fault_faultName") == "faultRegulation")
        .select("properties_payloadTimeStamp", "properties_deviceID", "payload_fault_state")
    )

    telem_only = [c for c in df_telemetry_800w.columns
                  if c not in ("properties_payloadTimeStamp", "properties_deviceID")]

    telem_part = df_telemetry_800w.select(
        "properties_payloadTimeStamp", "properties_deviceID",
        *telem_only,
        F.lit(None).cast("boolean").alias("payload_fault_state"),
        F.lit(True).alias("_is_telem"),
    )

    fault_part = df_fault.select(
        "properties_payloadTimeStamp", "properties_deviceID",
        *[F.lit(None).cast(dict(df_telemetry_800w.dtypes)[c]).alias(c)
          for c in telem_only],
        "payload_fault_state",
        F.lit(False).alias("_is_telem"),
    )

    w = Window.partitionBy("properties_deviceID").orderBy("properties_payloadTimeStamp")

    df = (
        telem_part.unionByName(fault_part)
        .withColumn("payload_fault_state",
                    F.last("payload_fault_state", ignorenulls=True).over(w))
        .filter(F.col("_is_telem"))
        .drop("_is_telem")
    )
    df = df.withColumn(
        "payload_fault_state", F.coalesce(F.col("payload_fault_state"), F.lit(False))
    )

    # Delta seconds
    df = (
        df
        .withColumn("prev_ts", F.lag("properties_payloadTimeStamp").over(w))
        .withColumn(
            "delta_seconds",
            F.when(
                F.col("prev_ts").isNull() |
                (F.col("properties_payloadTimeStamp").cast("long") - F.col("prev_ts").cast("long") < 0),
                None
            ).otherwise(
                F.col("properties_payloadTimeStamp").cast("long") - F.col("prev_ts").cast("long")
            ).cast("double")
        )
        .drop("prev_ts")
    )

    sensor_cols = [c for c in df.columns
                   if c not in {"properties_deviceID", "properties_payloadTimeStamp", "payload_fault_state"}]
    df = df.na.drop(subset=sensor_cols)

    # Device ID index
    device_ids_all = df.select("properties_deviceID").distinct().orderBy("properties_deviceID")
    device_ids_all = device_ids_all.withColumn(
        "deviceId_index", (F.dense_rank().over(Window.orderBy("properties_deviceID")) - 1).cast("double")
    )
    df = df.join(F.broadcast(device_ids_all), on="properties_deviceID", how="inner")

    print(f"Devices: {device_ids_all.count()}")

    # Horizon label 
    H_seconds = H_days * 86400
    w_horizon = (
        Window.partitionBy("properties_deviceID")
        .orderBy(F.col("properties_payloadTimeStamp").cast("long"))
        .rangeBetween(0, H_seconds)
    )
    df = df.withColumn(
        "label",
        F.max(F.col("payload_fault_state").cast("int")).over(w_horizon)
    )

    # Lag features 
    w_lag = Window.partitionBy("properties_deviceID").orderBy("properties_payloadTimeStamp")
    lag_exprs = [F.lag(c, L).over(w_lag).alias(f"{c}{L}")
                 for L in range(1, n_lags + 1) for c in sensor_cols]
    df = df.select("*", *lag_exprs)

    lag_cols = [f"{c}{L}" for L in range(1, n_lags + 1) for c in sensor_cols]
    df = df.na.drop(subset=lag_cols)

    print("Lag features computed.")

    # Lineage break
    temp_path = "/tmp/sensorx_retrain_lag"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(temp_path)
    df = spark.read.format("delta").load(temp_path)

    print("Lineage break complete.")

    # Deviation features
    w_daily = (
        Window.partitionBy("properties_deviceID")
        .orderBy(F.col("properties_payloadTimeStamp").cast("long"))
        .rangeBetween(-86400, 0)
    )
    dev_sensors = [
        "payload_xrayController_filamentCurrent",
        "payload_xrayController_temperature",
        "payload_xrayController_tubeCurrent",
    ]
    dev_cols = []
    dev_exprs = []
    for col_name in dev_sensors:
        avg_col = f"{col_name}_avg_daily"
        dev_col = f"{col_name}_dev_daily"
        dev_cols.extend([avg_col, dev_col])
        dev_exprs.append(F.avg(col_name).over(w_daily).alias(avg_col))
        dev_exprs.append((F.col(col_name) - F.avg(col_name).over(w_daily)).alias(dev_col))
    df = df.select("*", *dev_exprs)

    # Assemble features (112)
    feature_input_cols = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
    assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
    df_features = assembler.transform(df)
    df_features = df_features.select("properties_payloadTimeStamp", "features", "label")

    df_features = df_features.cache()
    row_count = df_features.count()
    vec_size = df_features.select("features").first()[0].size
    print(f"\nFeature engineering complete: {row_count:,} rows, {vec_size} features")
    assert vec_size == 112, f"Expected 112 features but got {vec_size}!"

    df_features.groupBy("label").count().orderBy("label").show()

    # Chronological train/test split (80/20)
    cutoff = df_features.selectExpr(
        "percentile_approx(CAST(properties_payloadTimeStamp AS BIGINT), 0.8)"
    ).first()[0]
    train = df_features.filter(F.col("properties_payloadTimeStamp").cast("bigint") <= cutoff)
    test = df_features.filter(F.col("properties_payloadTimeStamp").cast("bigint") > cutoff)

    train = train.drop("properties_payloadTimeStamp")
    test = test.drop("properties_payloadTimeStamp")

    return train, test

In [0]:
# Data pipeline parameters
days = 90
H_days = 15
n_lags = 20

# Training parameters (used in Cell 6)
layers = [112, 16, 16, 2]
max_iter = 100
step_size = 0.001

# Run data pipeline
train, test = data_engineering_retrain_mlp(days=days, H_days=H_days, n_lags=n_lags)

In [0]:
# Save tables
train.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("teams.sensorx.train_data")
test.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("teams.sensorx.test_data")
print("Saved teams.sensorx.train_data and test_data.")

In [0]:
# Normalize + Train MLP
scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train)
train_norm = scaler_model.transform(train).drop("features").withColumnRenamed("features_scaled", "features")
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

trainer = MultilayerPerceptronClassifier(
    featuresCol="features", labelCol="label",
    maxIter=max_iter, layers=layers, blockSize=128, seed=42, stepSize=step_size
)
print(f"\nTraining MLP {layers} (maxIter={max_iter}, stepSize={step_size})...")
model = trainer.fit(train_norm)
predictions = model.transform(test_norm)

# Evaluation
auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
recall_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="recallByLabel")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="fMeasureByLabel")

train_auc = auc_eval.evaluate(model.transform(train_norm))
test_auc = auc_eval.evaluate(predictions)
recall_1 = recall_eval.evaluate(predictions, {recall_eval.metricLabel: 1.0})
f1_1 = f1_eval.evaluate(predictions, {f1_eval.metricLabel: 1.0})

print(f"RESULTS")
print(f"  Training AUC: {train_auc:.4f}")
print(f"  Test AUC:     {test_auc:.4f}")
print(f"  Recall (fault): {recall_1:.4f}")
print(f"  F1 (fault):     {f1_1:.4f}")
predictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
# Register model 
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Shared/sensorx_mlp_fault_prediction")

MODEL_NAME = "teams.sensorx.mlp_fault_prediction"

with mlflow.start_run(run_name=f"MLP_retrain_{days}d_H{H_days}") as run:
    mlflow.log_param("model_type", "MultilayerPerceptronClassifier")
    mlflow.log_param("layers", str(layers))
    mlflow.log_param("maxIter", max_iter)
    mlflow.log_param("stepSize", step_size)
    mlflow.log_param("n_lags", n_lags)
    mlflow.log_param("H_days", H_days)
    mlflow.log_param("training_days", days)
    mlflow.log_param("num_features", 112)
    mlflow.log_param("train_rows", train.count())
    mlflow.log_param("normalized", True)
    mlflow.log_param("data_source", "prod_medallion.silver_machine.sensorx_telemetry")

    mlflow.log_metric("AUC", test_auc)
    mlflow.log_metric("Training_AUC", train_auc)
    mlflow.log_metric("Recall_label_1", recall_1)
    mlflow.log_metric("F1_label_1", f1_1)

    sample_input = test_norm.select("features").limit(5)
    sample_output = model.transform(sample_input).select("prediction")
    signature = infer_signature(sample_input.toPandas(), sample_output.toPandas())

    input_example = (
        sample_input.limit(1)
        .select(vector_to_array("features").alias("features"))
        .toPandas()
    )

    model_info = mlflow.spark.log_model(
        model, artifact_path="mlp_model",
        signature=signature, input_example=input_example,
        registered_model_name=MODEL_NAME,
    )

    print(f"\nModel registered: {MODEL_NAME}")
    print(f"  Version: {model_info.registered_model_version}")
    print(f"  Run ID:  {run.info.run_id}")
    print(f"  AUC:     {test_auc:.4f}")